# NHL Analytics Hub - Part 1: Exploring the NHL API

Before writing the loader, this notebook calls each endpoint and looks at the raw JSON: which fields we need, how they are nested, where values can be missing, and where the same record shows up more than once. Nothing is saved to disk - the responses only live in memory. The full fetch-and-load happens in `02_fetch_and_load.ipynb`.

Base URL: `https://api-web.nhle.com/v1` (no API key needed)

| Data | Endpoint |
|---|---|
| Teams + standings | `/standings/now` |
| Players | `/roster/{team}/current` |
| Games | `/club-schedule-season/{team}/20252026` |
| Skater + goalie season stats | `/club-stats/{team}/20252026/2` |
| Per-game skater stats | `/gamecenter/{game_id}/boxscore` |
| Players no longer on a roster | `/player/{player_id}/landing` |

In [1]:
import json
import time
from collections import Counter

import requests

BASE_URL = "https://api-web.nhle.com/v1"
SEASON = "20252026"
DELAY = 0.5  # seconds to wait before each request


def get_json(url, retries=3):
    for attempt in range(1, retries + 1):
        time.sleep(DELAY)
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as err:
            print(f"Attempt {attempt} failed for {url}: {err}")
            time.sleep(attempt * 2)
    return None


def show(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))

## 1. Standings (teams + standings tables)

In [2]:
standings = get_json(f"{BASE_URL}/standings/now")
teams = [row["teamAbbrev"]["default"] for row in standings["standings"]]

print("Top-level keys:", list(standings.keys()))
print("Teams found:", len(teams))
print(sorted(teams))

Top-level keys: ['wildCardIndicator', 'standingsDateTimeUtc', 'standings']
Teams found: 32
['ANA', 'BOS', 'BUF', 'CAR', 'CBJ', 'CGY', 'CHI', 'COL', 'DAL', 'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NJD', 'NSH', 'NYI', 'NYR', 'OTT', 'PHI', 'PIT', 'SEA', 'SJS', 'STL', 'TBL', 'TOR', 'UTA', 'VAN', 'VGK', 'WPG', 'WSH']


In [3]:
first = standings["standings"][0]
fields = ["seasonId", "date", "teamAbbrev", "teamName", "conferenceName", "divisionName", "teamLogo",
          "gamesPlayed", "wins", "losses", "otLosses", "points", "goalFor", "goalAgainst",
          "homeWins", "roadWins", "streakCode", "streakCount"]
for field in fields:
    print(f"{field:15} {first.get(field)!r}")

seasonId        20252026
date            '2026-04-17'
teamAbbrev      {'default': 'COL'}
teamName        {'default': 'Colorado Avalanche', 'fr': 'Avalanche du Colorado'}
conferenceName  'Western'
divisionName    'Central'
teamLogo        'https://assets.nhle.com/logos/nhl/svg/COL_light.svg'
gamesPlayed     82
wins            55
losses          16
otLosses        11
points          121
goalFor         302
goalAgainst     203
homeWins        26
roadWins        29
streakCode      'W'
streakCount     3


**What this tells us:**
- Team names are nested (`teamName = {"default": "Colorado Avalanche", "fr": ...}`), so we take `["default"]`.
- There is no numeric team id, only `teamAbbrev`, so `team_id` is an AUTO_INCREMENT key generated by MySQL.
- In the off-season `/standings/now` returns the final 2025-26 standings (`seasonId = 20252026`).
- `roadWins` maps to `away_wins`; `goalFor` / `goalAgainst` map to `goals_for` / `goals_against`.

## 2. Rosters (players table)

In [4]:
roster = get_json(f"{BASE_URL}/roster/COL/current")
print("Groups:", {group: len(players) for group, players in roster.items()})
show(roster["forwards"][0])

Groups: {'forwards': 20, 'defensemen': 12, 'goalies': 4}
{
  "id": 8481550,
  "headshot": "https://assets.nhle.com/mugs/nhl/20262027/COL/8481550.png",
  "firstName": {
    "default": "Adam"
  },
  "lastName": {
    "default": "Beckman"
  },
  "sweaterNumber": 34,
  "positionCode": "L",
  "shootsCatches": "L",
  "heightInInches": 74,
  "weightInPounds": 192,
  "heightInCentimeters": 188,
  "weightInKilograms": 87,
  "birthDate": "2001-05-10",
  "birthCity": {
    "default": "Saskatoon"
  },
  "birthCountry": "CAN",
  "birthStateProvince": {
    "default": "SK"
  }
}


In [5]:
# check every team's roster for position codes and missing values
needed = ["id", "firstName", "lastName", "positionCode", "sweaterNumber", "birthDate", "birthCountry",
          "heightInCentimeters", "weightInKilograms", "shootsCatches", "headshot"]
missing = Counter()
positions = Counter()
roster_ids = set()

for team in teams:
    data = get_json(f"{BASE_URL}/roster/{team}/current") or {}
    for group in data.values():
        for player in group:
            roster_ids.add(player["id"])
            positions[player.get("positionCode")] += 1
            for field in needed:
                if player.get(field) is None:
                    missing[field] += 1

print("Players on current rosters:", len(roster_ids))
print("Position codes:", dict(positions))
print("Missing values:", dict(missing) or "none")

Players on current rosters: 979
Position codes: {'L': 154, 'C': 270, 'R': 142, 'D': 311, 'G': 102}
Missing values: {'shootsCatches': 1, 'sweaterNumber': 20}


**What this tells us:**
- Players come in three lists (`forwards`, `defensemen`, `goalies`) that have to be flattened into one table.
- Position codes are `C`, `L`, `R`, `D`, `G`. The project spec uses `LW` / `RW`, so `L` and `R` get mapped.
- First and last names are nested like team names.
- Any field listed under "Missing values" needs a nullable column (e.g. `jersey_number`).

## 3. Team schedules (games table)

In [6]:
schedule = get_json(f"{BASE_URL}/club-schedule-season/COL/{SEASON}")
game = schedule["games"][0]
for field in ["id", "season", "gameType", "gameDate", "gameState", "venue", "homeTeam", "awayTeam"]:
    print(f"{field:10} {game.get(field)!r}")

id         2025010103
season     20252026
gameType   1
gameDate   '2025-09-21'
gameState  'FINAL'
venue      {'default': 'Magness Arena'}
homeTeam   {'id': 68, 'commonName': {'default': 'Mammoth'}, 'placeName': {'default': 'Utah'}, 'placeNameWithPreposition': {'default': 'Utah', 'fr': "de l'Utah"}, 'abbrev': 'UTA', 'logo': 'https://assets.nhle.com/logos/nhl/svg/UTA_light.svg?season=20252026', 'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/UTA_dark.svg?season=20252026', 'homeSplitSquad': True, 'score': 1}
awayTeam   {'id': 21, 'commonName': {'default': 'Avalanche'}, 'placeName': {'default': 'Colorado'}, 'placeNameWithPreposition': {'default': 'Colorado', 'fr': 'du Colorado'}, 'abbrev': 'COL', 'logo': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg', 'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/COL_dark.svg', 'awaySplitSquad': True, 'score': 5}


In [7]:
all_games = {}
rows_seen = 0
for team in teams:
    data = get_json(f"{BASE_URL}/club-schedule-season/{team}/{SEASON}") or {"games": []}
    rows_seen += len(data["games"])
    for g in data["games"]:
        all_games[g["id"]] = g

print("Game rows across all 32 schedules:", rows_seen)
print("Unique games:", len(all_games))
print("By (gameType, gameState):", dict(Counter((g["gameType"], g["gameState"]) for g in all_games.values())))

outside = Counter()
for g in all_games.values():
    for side in ("homeTeam", "awayTeam"):
        if g[side]["abbrev"] not in teams:
            outside[g[side]["abbrev"]] += 1
print("Opponents that are not one of the 32 NHL teams:", dict(outside) or "none")

Game rows across all 32 schedules: 2996
Unique games: 1498
By (gameType, gameState): {(1, 'FINAL'): 104, (2, 'OFF'): 1312, (3, 'OFF'): 82}
Opponents that are not one of the 32 NHL teams: none


**What this tells us:**
- Each game appears in both teams' schedules, so the raw row count is about double the unique count. `game_id` is the primary key and `INSERT IGNORE` skips the second copy.
- `gameType`: 1 = preseason, 2 = regular season, 3 = playoffs.
- `gameState` is `OFF` / `FINAL` for finished games and `FUT` for upcoming ones. Scores only exist once a game is played.
- The venue is nested (`venue = {"default": ...}`).
- An opponent that isn't one of the 32 teams can't satisfy the `teams` foreign key, so such games are skipped.

## 4. Club stats (skater_season_stats + goalie_season_stats)

In [8]:
club = get_json(f"{BASE_URL}/club-stats/COL/{SEASON}/2")
print("Keys:", list(club.keys()), "| skaters:", len(club["skaters"]), "| goalies:", len(club["goalies"]))
print("\nSkater example:")
show(club["skaters"][0])
print("\nGoalie example:")
show(club["goalies"][0])

Keys: ['season', 'gameType', 'skaters', 'goalies'] | skaters: 33 | goalies: 3

Skater example:
{
  "playerId": 8470613,
  "headshot": "https://assets.nhle.com/mugs/nhl/20252026/COL/8470613.png",
  "firstName": {
    "default": "Brent"
  },
  "lastName": {
    "default": "Burns"
  },
  "positionCode": "D",
  "gamesPlayed": 82,
  "goals": 12,
  "assists": 23,
  "points": 35,
  "plusMinus": 33,
  "penaltyMinutes": 32,
  "powerPlayGoals": 0,
  "shorthandedGoals": 0,
  "gameWinningGoals": 5,
  "overtimeGoals": 0,
  "shots": 172,
  "shootingPctg": 0.069767,
  "avgTimeOnIcePerGame": 1132.5732,
  "avgShiftsPerGame": 24.6098,
  "faceoffWinPctg": 0.0
}

Goalie example:
{
  "playerId": 8475809,
  "headshot": "https://assets.nhle.com/mugs/nhl/20252026/COL/8475809.png",
  "firstName": {
    "default": "Scott"
  },
  "lastName": {
    "default": "Wedgewood"
  },
  "gamesPlayed": 45,
  "gamesStarted": 43,
  "wins": 31,
  "losses": 6,
  "overtimeLosses": 6,
  "goalsAgainstAverage": 2.02427,
  "savePer

**What this tells us:**
- Skaters and goalies come back together. Skaters have a `positionCode`; goalies are position `G`, which routes them to `goalie_season_stats`.
- `avgTimeOnIcePerGame` is in **seconds**, but `avg_toi` should be `MM:SS`, so it gets converted.
- `savePercentage` is already a fraction and `saves` is given directly.
- `/2` means regular season. A player traded mid-season appears once for each team.

## 5. Boxscore (game_stats table)

In [9]:
finished = [g for g in schedule["games"] if g["gameState"] in ("OFF", "FINAL") and g["gameType"] == 2]
box = get_json(f"{BASE_URL}/gamecenter/{finished[0]['id']}/boxscore")

print("Keys:", list(box.keys()))
print("Home:", box["homeTeam"]["abbrev"], "| Away:", box["awayTeam"]["abbrev"])
print("playerByGameStats:", {side: list(groups.keys()) for side, groups in box["playerByGameStats"].items()})
print("\nForward example:")
show(box["playerByGameStats"]["homeTeam"]["forwards"][0])

Keys: ['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'regPeriods', 'awayTeam', 'homeTeam', 'clock', 'playerByGameStats', 'gameOutcome']
Home: LAK | Away: COL
playerByGameStats: {'awayTeam': ['forwards', 'defense', 'goalies'], 'homeTeam': ['forwards', 'defense', 'goalies']}

Forward example:
{
  "playerId": 8477960,
  "sweaterNumber": 9,
  "name": {
    "default": "A. Kempe"
  },
  "position": "R",
  "goals": 0,
  "assists": 1,
  "points": 1,
  "plusMinus": 0,
  "pim": 0,
  "hits": 5,
  "powerPlayGoals": 0,
  "sog": 2,
  "faceoffWinningPctg": 0.0,
  "toi": "19:09",
  "blockedShots": 1,
  "shifts": 21,
  "giveaways": 0,
  "takeaways": 1
}


**What this tells us:**
- Player lines sit under `playerByGameStats.homeTeam` / `awayTeam`, split into `forwards`, `defense` and `goalies`.
- `game_stats` is skaters only, so only `forwards` and `defense` are used; the team is whichever side the player is listed under.
- Mapping: `sog` -> `shots_on_goal`, `pim` -> `penalty_min`, `plusMinus` -> `plus_minus`; `toi` is already `MM:SS`.
- One call per finished game makes this the slowest part of the load (well over a thousand requests).

## 6. Players who are not on a current roster

The data is being collected in the off-season, so `/roster/{team}/current` is next season's roster. Comparing it with the 2025-26 club stats shows how many players with stats are missing from it.

In [10]:
stat_ids = [p["playerId"] for p in club["skaters"] + club["goalies"]]
not_on_roster = [pid for pid in stat_ids if pid not in roster_ids]
print(f"COL players with 2025-26 stats: {len(stat_ids)}, not on any current roster: {len(not_on_roster)}")

if not_on_roster:
    landing = get_json(f"{BASE_URL}/player/{not_on_roster[0]}/landing")
    for field in ["playerId", "firstName", "lastName", "position", "sweaterNumber", "birthDate", "birthCountry",
                  "heightInCentimeters", "weightInKilograms", "shootsCatches", "headshot"]:
        print(f"{field:20} {landing.get(field)!r}")

COL players with 2025-26 stats: 36, not on any current roster: 6


playerId             8476391
firstName            {'default': 'T.J.', 'cs': 'TJ', 'de': 'TJ', 'fi': 'TJ', 'sk': 'TJ', 'sv': 'TJ'}
lastName             {'default': 'Tynan'}
position             'C'
sweaterNumber        9
birthDate            '1992-02-25'
birthCountry         'USA'
heightInCentimeters  173
weightInKilograms    73
shootsCatches        'R'
headshot             'https://assets.nhle.com/mugs/nhl/latest/8476391.png'


**What this tells us:**
- Without these players, their stats rows would break the `players` foreign key.
- `/player/{id}/landing` has the same bio fields as the roster (with `position` instead of `positionCode`), so the loader uses it to add any player that shows up in the stats but isn't in `players` yet.